# Fase 2 — Modelagem Estocástica com SimPy
## Projeto de Redes de Computadores 2026-1 — PPGCC/UFPI

**Aluno:** Arthur Sabino Santos  
**Matrícula:** 20261005029  
**Orientador:** Prof. Rayner Gomes Sousa  

---

**Objetivo:** Desenvolver um simulador de eventos discretos (SimPy) que espelhe o comportamento do protocolo R-UDP/GBN implementado na Fase 1, executando 10 tarefas de validação e comparando os resultados com os dados reais.

In [ ]:
# Instalar dependências
!pip install -q simpy numpy matplotlib seaborn scipy kaleido==0.2.1 plotly

In [ ]:
import simpy
import heapq
import random
import json
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from typing import Dict, List, Any, Optional

sns.set_theme(style='whitegrid', palette='deep', font_scale=1.05)
print(f'SimPy {simpy.__version__} | NumPy {np.__version__}')

## 1. Núcleo do Simulador GBN (SimPy)

In [ ]:
# ── Constantes (idênticas à Fase 1: src/config.py) ───────────────────────────
CHUNK_SIZE     = 1400           # bytes por pacote
WINDOW_SIZE    = 8              # janela GBN
TIMEOUT_SEC    = 0.3            # timeout de retransmissão
FILE_SIZE_10MB = 10 * 1024 * 1024

# ── Cenários (espelham tc qdisc netem da Fase 1) ────────────────────────────
SCENARIOS = {
    'A': {'loss_prob': 0.00, 'delay_mean': 0.010, 'delay_std': 0.002},
    'B': {'loss_prob': 0.10, 'delay_mean': 0.050, 'delay_std': 0.005},
    'C': {'loss_prob': 0.20, 'delay_mean': 0.100, 'delay_std': 0.010},
    'S': {'loss_prob': 0.25, 'delay_mean': 0.100, 'delay_std': 0.010},
}
COLORS = {'A': '#2196F3', 'B': '#FF9800', 'C': '#F44336', 'S': '#9C27B0'}

print('Configuração do simulador:')
for sc, p in SCENARIOS.items():
    print(f'  Cenário {sc}: perda={p["loss_prob"]*100:.0f}%, '
          f'delay={p["delay_mean"]*1000:.0f}ms ± {p["delay_std"]*1000:.0f}ms')

In [ ]:
def simulate_gbn(
    file_size_bytes: int = FILE_SIZE_10MB,
    loss_prob: float = 0.0,
    delay_mean_s: float = 0.010,
    delay_std_s: float = 0.002,
    window_size: int = WINDOW_SIZE,
    timeout_s: float = TIMEOUT_SEC,
    seed: Optional[int] = None,
    return_rtt_samples: bool = False,
) -> Dict[str, Any]:
    """
    Simulação GBN (Go-Back-N) via SimPy — eventos discretos.

    Modelo de canal:
    - Perda de Bernoulli: replica tc qdisc netem loss da Fase 1
    - Atraso: one-way delay ~ N(delay_mean_s, delay_std_s²)
    - Receptor GBN: descarta pacotes fora de ordem (seq != expected_seq)

    Retorna dict com: transfer_time, throughput_mbps, retransmissions,
                      data_sent, acks_received, rtt_mean, rtt_std,
                      total_packets, efficiency
    """
    if seed is not None:
        random.seed(seed)
        np.random.seed(seed)

    total_pkts = max(1, file_size_bytes // CHUNK_SIZE)
    env = simpy.Environment()

    st: Dict[str, Any] = {
        'base': 0, 'next_seq': 0, 'retransmissions': 0,
        'data_sent': 0, 'acks_received': 0, 'rtt_samples': [],
        'ack_heap': [], 'recv_expected': 0,
    }

    def one_way_delay() -> float:
        d = random.gauss(delay_mean_s, max(1e-6, delay_std_s))
        return max(1e-6, d)

    def tx_packet(seq: int) -> None:
        st['data_sent'] += 1
        if random.random() < loss_prob:   # Modelo Bernoulli
            return
        if seq != st['recv_expected']:     # GBN: descarta fora de ordem
            return
        st['recv_expected'] += 1
        fwd = one_way_delay()
        bck = one_way_delay()
        arrive_time = env.now + fwd + bck
        st['rtt_samples'].append(fwd + bck)
        heapq.heappush(st['ack_heap'], (arrive_time, seq))

    def gbn_sender(env: simpy.Environment):
        """Processo SimPy do emissor GBN (usa yield env.timeout)."""
        timer_start = env.now
        h = st['ack_heap']

        while st['base'] < total_pkts:
            # Preencher janela
            while (st['next_seq'] < total_pkts and
                   st['next_seq'] < st['base'] + window_size):
                tx_packet(st['next_seq'])
                st['next_seq'] += 1

            # Descartar ACKs obsoletos
            while h and h[0][1] < st['base']:
                heapq.heappop(h)

            timeout_abs  = timer_start + timeout_s
            earliest_ack = h[0][0] if h else float('inf')
            next_event   = min(earliest_ack, timeout_abs)

            # Avançar relógio de simulação (SimPy)
            yield env.timeout(max(0.0, next_event - env.now))

            if earliest_ack <= timeout_abs and h:
                # ACK recebido — avança janela
                _, seq = heapq.heappop(h)
                now = env.now
                arrived = {seq}
                stale = []
                while h and h[0][0] <= now:
                    t2, s2 = heapq.heappop(h)
                    if s2 >= st['base']: arrived.add(s2)
                    else: stale.append((t2, s2))
                for item in stale: heapq.heappush(h, item)

                while st['base'] in arrived and st['base'] < total_pkts:
                    st['base'] += 1
                    st['acks_received'] += 1
                timer_start = env.now
            else:
                # Timeout — Go-Back-N: retransmite toda a janela
                st['ack_heap'] = [(t, s) for t, s in h if s < st['base']]
                heapq.heapify(st['ack_heap'])
                h = st['ack_heap']
                st['recv_expected'] = st['base']  # reset receptor
                for s in range(st['base'], st['next_seq']):
                    tx_packet(s)
                    st['retransmissions'] += 1
                timer_start = env.now

    env.process(gbn_sender(env))
    env.run()

    t   = env.now
    rtt = st['rtt_samples']
    result = {
        'transfer_time':   t,
        'throughput_mbps': (file_size_bytes / t / 1e6) if t > 0 else 0.0,
        'retransmissions': st['retransmissions'],
        'data_sent':       st['data_sent'],
        'acks_received':   st['acks_received'],
        'rtt_mean':        float(np.mean(rtt)) if rtt else 2 * delay_mean_s,
        'rtt_std':         float(np.std(rtt))  if len(rtt) > 1 else 0.0,
        'total_packets':   total_pkts,
        'efficiency':      total_pkts / st['data_sent'] if st['data_sent'] > 0 else 0.0,
    }
    if return_rtt_samples:
        result['rtt_samples'] = rtt
    return result


def run_multiple(n_runs=30, seed_start=42, **sim_kwargs):
    keys = ['transfer_time','throughput_mbps','retransmissions',
            'data_sent','acks_received','rtt_mean','rtt_std','total_packets','efficiency']
    res = {k: [] for k in keys}
    for i in range(n_runs):
        r = simulate_gbn(seed=seed_start + i, **sim_kwargs)
        for k in keys: res[k].append(float(r[k]))
    return res


def stats_ci95(values):
    arr = np.asarray(values, dtype=float)
    n = len(arr)
    mu  = float(np.mean(arr))
    std = float(np.std(arr, ddof=1)) if n > 1 else 0.0
    ci  = 1.96 * std / np.sqrt(n)   if n > 0 else 0.0
    return {'mean': mu, 'std': std, 'ci95': ci, 'n': n}


# Teste rápido
r = simulate_gbn(seed=42, **SCENARIOS['A'])
print(f'Cenário A — Vazão: {r["throughput_mbps"]:.4f} Mbps | '
      f'Tempo: {r["transfer_time"]:.2f}s | Retrans: {r["retransmissions"]}')

## 2. Dados Reais da Fase 1

In [ ]:
# Métricas reais (rudp_client_metrics.jsonl)
REAL_RUDP = {
    'A': {
        'transfer_times':  [10.9207, 10.6400, 10.6580, 10.5320, 10.6309],
        'throughputs':     [7.6814,  7.8840,  7.8707,  7.9649,  7.8907],
        'retransmissions': [8, 0, 0, 0, 0],
    },
    'B': {
        'transfer_times':  [325.3516, 312.7303, 315.8990, 318.0697, 315.9218],
        'throughputs':     [0.2578,   0.2682,   0.2655,   0.2637,   0.2655],
        'retransmissions': [6832, 6536, 6630, 6651, 6611],
    },
    'C': {
        'transfer_times':  [758.2228, 741.9798, 761.9748],
        'throughputs':     [0.1106,   0.1131,   0.1101],
        'retransmissions': [15276, 14890, 15488],
    },
}

print('Resumo das métricas reais (Fase 1):')
for sc in ['A','B','C']:
    tp = np.mean(REAL_RUDP[sc]['throughputs'])
    tt = np.mean(REAL_RUDP[sc]['transfer_times'])
    rt = np.mean(REAL_RUDP[sc]['retransmissions'])
    print(f'  Cenário {sc}: {tp:.4f} Mbps | {tt:.2f}s | {rt:.0f} retrans')

## 3. Tarefas de Validação

### Tarefa 1 — Modelagem de Atraso (Distribuição Normal)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
scenarios_info = [
    ('A', 'Cenário A (10ms)', 0.010, 0.002),
    ('B', 'Cenário B (50ms)', 0.050, 0.005),
    ('C', 'Cenário C (100ms)', 0.100, 0.010),
]
np.random.seed(42)
for ax, (sc, title, mu, sigma) in zip(axes, scenarios_info):
    n = 5000
    fwd = np.maximum(1e-6, np.random.normal(mu, sigma, n))
    bck = np.maximum(1e-6, np.random.normal(mu, sigma, n))
    rtt_ms = (fwd + bck) * 1000
    ax.hist(rtt_ms, bins=50, density=True, alpha=0.6, color=COLORS[sc],
            label='Amostras simuladas', edgecolor='white')
    mu_t = 2 * mu * 1000; sigma_t = np.sqrt(2) * sigma * 1000
    x = np.linspace(rtt_ms.min(), rtt_ms.max(), 300)
    ax.plot(x, stats.norm.pdf(x, mu_t, sigma_t), 'r-', lw=2.5,
            label=f'N({mu_t:.0f}, {sigma_t:.2f})')
    ax.axvline(mu_t, color='r', linestyle='--', alpha=0.4)
    ax.set_title(title); ax.set_xlabel('RTT (ms)'); ax.set_ylabel('Densidade')
    ax.legend(fontsize=8)
plt.suptitle('Tarefa 1: Modelagem de Atraso — N(μ,σ²)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('task1_delay_modeling.png', dpi=150, bbox_inches='tight')
plt.show()
print('Tarefa 1 concluída.')

### Tarefa 2 — Modelo de Perda de Bernoulli

In [ ]:
print('Executando Tarefa 2 (Bernoulli)...')
configs = [0.00, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30]
obs_means, obs_stds = [], []
for p in configs:
    losses = []
    for seed in range(10):
        r = simulate_gbn(file_size_bytes=3*1024*1024, loss_prob=p,
                         delay_mean_s=0.050, delay_std_s=0.005, seed=seed)
        losses.append(r['retransmissions']/r['data_sent'] if r['data_sent']>0 else 0)
    obs_means.append(np.mean(losses)); obs_stds.append(np.std(losses))

fig, ax = plt.subplots(figsize=(9,5))
x = np.array(configs)
ax.errorbar(x*100, np.array(obs_means)*100, yerr=np.array(obs_stds)*100,
            fmt='o-', color='#2196F3', lw=2, ms=7, capsize=5, label='Overhead observado')
ax.plot(x*100, x*100, 'r--', lw=2, label='Referência')
ax.set_xlabel('Taxa de perda configurada (%)'); ax.set_ylabel('Overhead de retransmissão (%)')
ax.set_title('Tarefa 2: Modelo de Perda de Bernoulli'); ax.legend()
plt.tight_layout()
plt.savefig('task2_bernoulli_loss.png', dpi=150, bbox_inches='tight')
plt.show()
print('Tarefa 2 concluída.')

### Tarefa 3 — Simulação de Timeout (Retransmissões Real vs Simulado)

In [ ]:
print('Executando Tarefa 3 (Timeout)...')
scenarios_abc = ['A','B','C']
real_means=[]; real_stds=[]; sim_means=[]; sim_stds=[]
for sc in scenarios_abc:
    real_r = REAL_RUDP[sc]['retransmissions']
    real_means.append(np.mean(real_r)); real_stds.append(np.std(real_r))
    sim_r = [simulate_gbn(seed=s, **SCENARIOS[sc])['retransmissions'] for s in range(10)]
    sim_means.append(np.mean(sim_r)); sim_stds.append(np.std(sim_r))
    print(f'  Cenário {sc}: Real={np.mean(real_r):.0f} | Sim={np.mean(sim_r):.0f}')

fig, ax = plt.subplots(figsize=(9,6))
x = np.arange(3); w = 0.35
ax.bar(x-w/2, real_means, w, yerr=real_stds, capsize=5, label='Real (Fase 1)', color='#2196F3', alpha=0.85)
ax.bar(x+w/2, sim_means,  w, yerr=sim_stds,  capsize=5, label='Simulado (SimPy)', color='#FF9800', alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels([f'Cenário {s}' for s in scenarios_abc])
ax.set_ylabel('Retransmissões'); ax.set_title('Tarefa 3: Retransmissões Real vs Simulado'); ax.legend()
plt.tight_layout()
plt.savefig('task3_timeout_simulation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Tarefa 3 concluída.')

### Tarefa 4 — Curva de Vazão (1MB a 100MB)

In [ ]:
print('Executando Tarefa 4 (Curva de Vazão)...')
file_sizes_mb = [1,2,5,10,20,50,100]
file_sizes_b  = [s*1024*1024 for s in file_sizes_mb]

fig, ax = plt.subplots(figsize=(10,6))
for sc in ['A','B','C']:
    tp_m, tp_s = [], []
    for fsz in file_sizes_b:
        print(f'  Cenário {sc}, {fsz//1024//1024}MB...', end=' ')
        tp = [simulate_gbn(file_size_bytes=fsz, seed=s, **SCENARIOS[sc])['throughput_mbps'] for s in range(5)]
        tp_m.append(np.mean(tp)); tp_s.append(np.std(tp))
        print(f'{np.mean(tp):.4f} Mbps')
    ax.errorbar(file_sizes_mb, tp_m, yerr=tp_s, fmt='o-', color=COLORS[sc], lw=2, ms=6, capsize=4, label=f'Cenário {sc}')

ax.set_xscale('log'); ax.set_xlabel('Tamanho do Arquivo (MB)')
ax.set_ylabel('Vazão (Mbps)'); ax.set_title('Tarefa 4: Curva de Vazão — 1MB a 100MB')
ax.set_xticks(file_sizes_mb); ax.set_xticklabels([f'{s}MB' for s in file_sizes_mb])
ax.legend(); plt.tight_layout()
plt.savefig('task4_throughput_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print('Tarefa 4 concluída.')

### Tarefa 5 — Sensibilidade da Janela

In [ ]:
print('Executando Tarefa 5 (Sensibilidade da Janela)...')
window_sizes = [1,2,4,8,16,32,64]
fig, axes = plt.subplots(1,2,figsize=(14,6))

for ax, (sc, label) in zip(axes, [('A','0%/10ms'),('B','10%/50ms')]):
    p = SCENARIOS[sc]
    tp_m, tp_s = [], []
    for W in window_sizes:
        tp = [simulate_gbn(file_size_bytes=5*1024*1024, window_size=W, seed=s, **p)['throughput_mbps']
              for s in range(5)]
        tp_m.append(np.mean(tp)); tp_s.append(np.std(tp))
    ax.errorbar(window_sizes, tp_m, yerr=tp_s, fmt='o-', color=COLORS[sc], lw=2, ms=7, capsize=4)
    ax.axvline(WINDOW_SIZE, color='gray', linestyle='--', alpha=0.6, label=f'W={WINDOW_SIZE} (Fase 1)')
    ax.set_xscale('log', base=2)
    ax.set_xticks(window_sizes); ax.set_xticklabels([str(w) for w in window_sizes])
    ax.set_xlabel('Tamanho da Janela W'); ax.set_ylabel('Vazão (Mbps)')
    ax.set_title(f'Cenário {sc}: {label}'); ax.legend()

plt.suptitle('Tarefa 5: Sensibilidade da Janela', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('task5_window_sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()
print('Tarefa 5 concluída.')

### Tarefa 6 — Validação de RTT

In [ ]:
print('Executando Tarefa 6 (RTT)...')
scenarios_abc = ['A','B','C']
rtt_config=[]; rtt_sim=[]; rtt_sim_std=[]; rtt_real=[]

for sc in scenarios_abc:
    p = SCENARIOS[sc]
    rtt_config.append(2 * p['delay_mean'] * 1000)
    rtts = [simulate_gbn(seed=s, **p)['rtt_mean']*1000 for s in range(10)]
    rtt_sim.append(np.mean(rtts)); rtt_sim_std.append(np.std(rtts))
    tp_mean = np.mean(REAL_RUDP[sc]['throughputs']) * 1e6
    rtt_real.append((WINDOW_SIZE * CHUNK_SIZE * 8) / tp_mean * 1000)
    print(f'  Cenário {sc}: config={rtt_config[-1]:.0f}ms | real={rtt_real[-1]:.1f}ms | sim={rtt_sim[-1]:.1f}ms')

fig, ax = plt.subplots(figsize=(9,6))
x = np.arange(3); w = 0.26
ax.bar(x-w, rtt_config, w, label='Configurado (2×tc)', color='#9E9E9E', alpha=0.85)
ax.bar(x,   rtt_real,   w, label='Real (implícito, Fase 1)', color='#2196F3', alpha=0.85)
ax.bar(x+w, rtt_sim,    w, label='Simulado (SimPy)', color='#FF9800', alpha=0.85)
ax.errorbar(x+w, rtt_sim, yerr=rtt_sim_std, fmt='none', color='black', capsize=4)
ax.set_xticks(x); ax.set_xticklabels([f'Cenário {s}' for s in scenarios_abc])
ax.set_ylabel('RTT (ms)'); ax.set_title('Tarefa 6: Validação de RTT'); ax.legend()
plt.tight_layout()
plt.savefig('task6_rtt_validation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Tarefa 6 concluída.')

### Tarefa 7 — Impacto do Jitter

In [ ]:
print('Executando Tarefa 7 (Jitter)...')
jitter_ms = [0,2,5,10,20,30,50]
tp_m=[]; tp_s=[]; rt_m=[]; rt_s=[]

for j in jitter_ms:
    tp=[]; rt=[]
    for seed in range(10):
        r = simulate_gbn(file_size_bytes=3*1024*1024, loss_prob=0.10,
                         delay_mean_s=0.050, delay_std_s=j/1000, seed=seed)
        tp.append(r['throughput_mbps']); rt.append(r['retransmissions'])
    tp_m.append(np.mean(tp)); tp_s.append(np.std(tp))
    rt_m.append(np.mean(rt)); rt_s.append(np.std(rt))

fig, (ax1,ax2) = plt.subplots(1,2,figsize=(14,5))
ax1.errorbar(jitter_ms, tp_m, yerr=tp_s, fmt='o-', color='#2196F3', lw=2, ms=6, capsize=4)
ax1.axvline(5, color='gray', linestyle='--', alpha=0.5, label='Jitter padrão (5ms)')
ax1.set_xlabel('Jitter (ms)'); ax1.set_ylabel('Vazão (Mbps)'); ax1.set_title('Vazão vs Jitter'); ax1.legend()

ax2.errorbar(jitter_ms, rt_m, yerr=rt_s, fmt='s-', color='#F44336', lw=2, ms=6, capsize=4)
ax2.axvline(5, color='gray', linestyle='--', alpha=0.5)
ax2.set_xlabel('Jitter (ms)'); ax2.set_ylabel('Retransmissões'); ax2.set_title('Retransmissões vs Jitter')

plt.suptitle('Tarefa 7: Impacto do Jitter (delay=50ms, perda=10%)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('task7_jitter_impact.png', dpi=150, bbox_inches='tight')
plt.show()
print('Tarefa 7 concluída.')

### Tarefa 8 — Cenário de Estresse (25% de Perda)

In [ ]:
print('Executando Tarefa 8 (Estresse 25%)...')
all_sc = ['A','B','C','S']
sc_labels = {'A':'A\n(0%/10ms)','B':'B\n(10%/50ms)','C':'C\n(20%/100ms)','S':'Stress\n(25%/100ms)'}
sim_tp={}; sim_tt={}

for sc in all_sc:
    tp=[]; tt=[]
    for seed in range(10):
        r = simulate_gbn(seed=seed, **SCENARIOS[sc])
        tp.append(r['throughput_mbps']); tt.append(r['transfer_time'])
    sim_tp[sc]=(np.mean(tp),np.std(tp)); sim_tt[sc]=(np.mean(tt),np.std(tt))
    print(f'  Cenário {sc}: {np.mean(tp):.4f} Mbps | {np.mean(tt):.1f}s')

fig,(ax1,ax2) = plt.subplots(1,2,figsize=(14,6))
x = np.arange(4); colors_list=[COLORS[sc] for sc in all_sc]
tp_means=[sim_tp[sc][0] for sc in all_sc]; tp_stds=[sim_tp[sc][1] for sc in all_sc]
tt_means=[sim_tt[sc][0] for sc in all_sc]; tt_stds=[sim_tt[sc][1] for sc in all_sc]

bars = ax1.bar(x, tp_means, yerr=tp_stds, capsize=5, color=colors_list, alpha=0.85)
ax1.set_xticks(x); ax1.set_xticklabels([sc_labels[sc] for sc in all_sc])
ax1.set_ylabel('Vazão (Mbps)'); ax1.set_title('Vazão por Cenário')
for bar,v in zip(bars,tp_means): ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()*1.05, f'{v:.3f}', ha='center', fontsize=8)

ax2.bar(x, tt_means, yerr=tt_stds, capsize=5, color=colors_list, alpha=0.85)
ax2.set_xticks(x); ax2.set_xticklabels([sc_labels[sc] for sc in all_sc])
ax2.set_ylabel('Tempo de Transferência (s)'); ax2.set_title('Tempo de Transferência')

plt.suptitle('Tarefa 8: Cenário de Estresse — 25% de Perda', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('task8_stress_scenario.png', dpi=150, bbox_inches='tight')
plt.show()
print('Tarefa 8 concluída.')

### Tarefa 9 — Análise de Eficiência

In [ ]:
print('Executando Tarefa 9 (Eficiência)...')
all_sc = ['A','B','C','S']
eff_sim={}; eff_real={}

for sc in all_sc:
    effs = [simulate_gbn(seed=s, **SCENARIOS[sc])['efficiency'] for s in range(15)]
    eff_sim[sc]=(np.mean(effs), np.std(effs))
    if sc in REAL_RUDP:
        total=7490
        er = [total/(total+r) for r in REAL_RUDP[sc]['retransmissions']]
        eff_real[sc]=(np.mean(er), np.std(er))
    print(f'  Cenário {sc}: Sim={np.mean(effs):.4f} | Real={eff_real.get(sc,(None,None))[0]}')

fig, ax = plt.subplots(figsize=(10,6))
x = np.arange(4); w=0.35
sim_m=[eff_sim[sc][0] for sc in all_sc]; sim_s=[eff_sim[sc][1] for sc in all_sc]
ax.bar(x+w/2, sim_m, w, yerr=sim_s, capsize=5, label='Simulado (SimPy)', color='#FF9800', alpha=0.85)
real_sc_list=[sc for sc in all_sc if sc in eff_real]
real_x=np.array([all_sc.index(sc) for sc in real_sc_list])
real_m=[eff_real[sc][0] for sc in real_sc_list]; real_s=[eff_real[sc][1] for sc in real_sc_list]
ax.bar(real_x-w/2, real_m, w, yerr=real_s, capsize=5, label='Real (Fase 1)', color='#2196F3', alpha=0.85)
ax.axhline(1.0, color='gray', linestyle='--', alpha=0.5, label='Ideal')
ax.set_xticks(x); ax.set_xticklabels([f'Cenário {s}' for s in all_sc])
ax.set_ylabel('Eficiência (pkts úteis / total enviados)'); ax.set_ylim(0,1.15)
ax.set_title('Tarefa 9: Análise de Eficiência'); ax.legend()
plt.tight_layout()
plt.savefig('task9_efficiency.png', dpi=150, bbox_inches='tight')
plt.show()
print('Tarefa 9 concluída.')

### Tarefa 10 — Convergência Estatística (IC 95%, 30 execuções)

In [ ]:
print('Executando Tarefa 10 (Convergência Estatística — 30 runs por cenário)...')
n_runs = 30
scenarios_abc = ['A','B','C']
fig, axes = plt.subplots(2,3,figsize=(16,10))

for col, sc in enumerate(scenarios_abc):
    print(f'  Cenário {sc}: {n_runs} runs...')
    res = run_multiple(n_runs=n_runs, seed_start=100, **SCENARIOS[sc])
    tps = np.array(res['throughput_mbps'])
    rts = np.array(res['retransmissions'])

    for ax, data, ylabel in [(axes[0,col], tps, 'Vazão (Mbps)'), (axes[1,col], rts, 'Retransmissões')]:
        cum_m = np.cumsum(data) / np.arange(1, n_runs+1)
        cum_s = [np.std(data[:k+1], ddof=1) if k>0 else 0 for k in range(n_runs)]
        cum_ci = np.array([1.96*s/np.sqrt(k+1) for k,s in enumerate(cum_s)])
        run_idx = np.arange(1, n_runs+1)
        ax.plot(run_idx, cum_m, 'b-', lw=2, label='Média cumulativa')
        ax.fill_between(run_idx, cum_m-cum_ci, cum_m+cum_ci, alpha=0.3, color='blue', label='IC 95%')
        ax.axhline(cum_m[-1], color='r', linestyle='--', alpha=0.6)
        ax.set_title(f'Cenário {sc} — {ylabel}'); ax.set_xlabel('Execuções'); ax.set_ylabel(ylabel)
        ax.legend(fontsize=7)

    st_tp = stats_ci95(tps); st_rt = stats_ci95(rts)
    print(f'    Vazão: {st_tp["mean"]:.4f} ± {st_tp["ci95"]:.4f} Mbps')
    print(f'    Retrans: {st_rt["mean"]:.1f} ± {st_rt["ci95"]:.1f}')

plt.suptitle('Tarefa 10: Convergência Estatística — IC 95% com 30 Execuções', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('task10_convergence.png', dpi=150, bbox_inches='tight')
plt.show()
print('Tarefa 10 concluída.')

## 4. Análise Comparativa: Real vs Simulado

In [ ]:
print('Gerando Comparação Real vs Simulado...')
scenarios_abc = ['A','B','C']
n_runs = 15

real_data = {
    'tp': {sc: (np.mean(REAL_RUDP[sc]['throughputs']), np.std(REAL_RUDP[sc]['throughputs'])) for sc in scenarios_abc},
    'tt': {sc: (np.mean(REAL_RUDP[sc]['transfer_times']), np.std(REAL_RUDP[sc]['transfer_times'])) for sc in scenarios_abc},
    'rt': {sc: (np.mean(REAL_RUDP[sc]['retransmissions']), np.std(REAL_RUDP[sc]['retransmissions'])) for sc in scenarios_abc},
}
sim_data = {'tp':{}, 'tt':{}, 'rt':{}}
for sc in scenarios_abc:
    tp=[]; tt=[]; rt=[]
    for seed in range(n_runs):
        r = simulate_gbn(seed=seed, **SCENARIOS[sc])
        tp.append(r['throughput_mbps']); tt.append(r['transfer_time']); rt.append(r['retransmissions'])
    sim_data['tp'][sc]=(np.mean(tp),np.std(tp))
    sim_data['tt'][sc]=(np.mean(tt),np.std(tt))
    sim_data['rt'][sc]=(np.mean(rt),np.std(rt))
    print(f'  Cenário {sc}: Real={real_data["tp"][sc][0]:.4f} Mbps | Sim={sim_data["tp"][sc][0]:.4f} Mbps')

fig, axes = plt.subplots(1,3,figsize=(16,6))
x = np.arange(3); w=0.35
labels_x = [f'Cenário {s}' for s in scenarios_abc]
metrics_info = [
    ('Vazão (Mbps)', 'tp'),
    ('Tempo de Transferência (s)', 'tt'),
    ('Retransmissões', 'rt'),
]
for ax, (ylabel, key) in zip(axes, metrics_info):
    rm=[real_data[key][sc][0] for sc in scenarios_abc]; rs=[real_data[key][sc][1] for sc in scenarios_abc]
    sm=[sim_data[key][sc][0]  for sc in scenarios_abc]; ss=[sim_data[key][sc][1]  for sc in scenarios_abc]
    ax.bar(x-w/2, rm, w, yerr=rs, capsize=5, label='Real (Fase 1)', color='#2196F3', alpha=0.85)
    ax.bar(x+w/2, sm, w, yerr=ss, capsize=5, label='Simulado (SimPy)', color='#FF9800', alpha=0.85)
    ax.set_xticks(x); ax.set_xticklabels(labels_x, fontsize=9)
    ax.set_ylabel(ylabel); ax.set_title(ylabel); ax.legend(fontsize=8)

plt.suptitle('Análise Comparativa: Real (Fase 1) vs Simulado (SimPy)\nGBN W=8, T=0.3s, 10MB',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('real_vs_simulated.png', dpi=150, bbox_inches='tight')
plt.show()
print('Comparação concluída.')

## 5. Sumário dos Resultados

| Tarefa | Descrição | Status |
|--------|-----------|--------|
| 1 | Modelagem de Atraso (N(μ,σ²)) | ✅ |
| 2 | Modelo de Perda de Bernoulli | ✅ |
| 3 | Simulação de Timeout (retransmissões) | ✅ |
| 4 | Curva de Vazão (1MB–100MB) | ✅ |
| 5 | Sensibilidade da Janela W | ✅ |
| 6 | Validação de RTT | ✅ |
| 7 | Impacto do Jitter | ✅ |
| 8 | Cenário de Estresse (25% perda) | ✅ |
| 9 | Análise de Eficiência | ✅ |
| 10 | Convergência Estatística (IC 95%, 30 runs) | ✅ |

**Conclusão:** O simulador SimPy reproduz os padrões qualitativos do sistema real R-UDP/GBN da Fase 1. As discrepâncias quantitativas observadas (especialmente em throughput) refletem diferenças entre o modelo de canal bidirecional do simulador e a aplicação unidirecional do tc qdisc no ambiente Docker real.

In [ ]:
print('\n' + '='*60)
print('FASE 2 COMPLETA — 10 Tarefas de Validação Concluídas')
print('Aluno: Arthur Sabino Santos (20261005029)')
print('PPGCC/UFPI 2026-1')
print('='*60)